# WEL01 · M07 검토·승인 상태와 담당 책임 확인 (교수자·해설용)

- 본 노트북은 **미션7(M07) 전용 교수자·해설용** 노트북입니다 (`ROLE = 'solution'`).
- 원자료 생성·참고 구현·상황판 소스가 노트북에 내장되어 있어 별도의 `.md` 파일 없이 위에서부터 바로 실행할 수 있습니다.
- 각 STEP 코드 셀에는 모범 pandas 풀이, 경계 조건 예외 처리, 해설 주석이 포함되어 있습니다.

In [ ]:
from pathlib import Path
import base64,zlib,json,re,html,hashlib,io,time,ast,copy
import numpy as np,pandas as pd
from IPython.display import display,HTML

PROJECT_ID='WEL01'
ROLE='solution'
WORKDIR=Path('/content') if Path('/content').is_dir() else Path.cwd()
PROJECT_HOME=WORKDIR/PROJECT_ID
OUTDIR=PROJECT_HOME/'outputs'
SOURCE_BUNDLE=None
DASHBOARD_ASSETS=None
REFERENCE_CACHE={}
REFERENCE_EXECUTION_SERIAL=0

def _sha(value):
    return hashlib.sha256(json.dumps(value,ensure_ascii=False,sort_keys=True,separators=(',',':')).encode()).hexdigest()

EMBEDDED_BUNDLE_B64 = 'eJztfftzE1eW/7/S46ktSZlGyE+MiVJFwCT+Lo9ZbHZnynap2lLL7kGvtNoGF8W3DBGsB5xvzAQHkcgeM0MCZJ0dAQZMhexW5U/ZHy35f/iex73dt1ttY3vIThLEPCy1bt977rnn8Tnnnr59qaNkF/9gpp2hTMdAx78Nnkx0dugdM6ZdtooFuHLBzCU6D1gFx5y0DcfMHOiCn23zo2nLNjMfFqftcsdAdzzhXTtlFaYdE6529sNVo1w2nRNWYdK0Szb0Aj1O9PZ0dfYeNrv6J/o6E4e6eiayxuFMV+dEeiKR6M9mJiY6s12ZzsNd2cwh00wfTiT6ursyRmdvz0RnX9bowfGnC46VN4dh+LQJXY7Rf4h47dTx5p1FbevTta2lh43FVW3z2WKzdq/5xSOtWd/YfLqqNW/e3/r8j5sb9ebdOa3xSb25vL5VqWvNz9cbC7fgU5x7GytY+VLRdrR0ESZ/0clZE7pV1MuzZR3H1v9QLhb0KaM8hT+ki6VZvZQznGzRzut8I1yP503HyBiO4XZWmM6XZjWjrBVKeskoZOAT/LeUGSuMFTJmVkvZZta0zULaTDnGRM6M5q2MDiRkTF1zpgtmJnm6WDBjA2MFDf6JTvOGU8oVacDSLH6iPnMONypOO6VpJ2kV48MOLMHk0Jlo7Ai2KidHx0WTXCZVnipeSMJNcfzAl5GgtFFypm2Tfo6+Y9iTZf2dd85fwA+SCoWSCaNs9vV4l4EdOGXNKiA98UnTSWWtSbhSjqp3U1NrkoaHvzBeFNrEjkxMZ5Hu92dBoIhs+DFeNmZM+BuFH3Vkt+EkI6XCZETPlKzk4YQ+MVG8mLIK6SmznIw41uSUE4n5h6LJx41SySxkohFcnwErb0yaB6GbIzwFPfIb/hCf6OuB5YAFwAFxBjNGbtqMxmLxjEmXY0rvOIF0rlg2oxEjl5PjSq4mVWbyT4Vy8lIklSoYeTOVigzAR1I4RQxSER2uZq0cNchGSMpTl0AuLnutYN0jl4VMZFlQNAuEDEQB5WWgUB6NjJw7PXg8dezM6RNDH0TGk9SIb3ESSRTpeMm0s6l0EZTLtFFGbHPGKk6XU45tpM0kSD5On75Excwce1ZZxguWM6WoSxysAViEtJMqOxmQwihLYkzfpolp27JJQDbMi2Y6mi7mS8CFKCkDTsjlyrgewRaRGFxWFsM2LuiogMkU6GDJ8bhFSuU2zVoFWCt1HjjTspypnwuoOWIxpdIcCV90GD0J/4MZQlcgjxnzYjRjF0vJEXvajMXRYkS9pnGrUDZtJ5rQI80vF8E+DR2P6KPZCK3zgUvWbzoHEn2ZyxHSKAv1yTbAqEZzZiEKt8di46IvnHB8ugRCbUY9ISqDqBYy5aQNi5uJhiz2ASeh98R0hXnuvchvmKjR1duXFPYuzl9pKeJCO0AhpsyLGWvSLDtRtSde/CSvrKJAOtsg+n89D10kI8fOnT07eHokdXbwxCB8ODaYGvzd4LFzI0NnTkeUHrFxLmVMO1NFuzxllZKR5mpt8+mzxmJNGvnGNw8bn2wIWw8WvvGorm0t3W3eeNZcWWwuV2WDxqfXGl9/D05ibataaS5XtObXy41vrm0tVZsP5rTm0o3m8oI6NBqLVBkWD6j1PMvW0qNm5TGMVmtcrTa+XmtW76NzgT9btyuN5VfNZRjpZQVvYFKaK5Xmx1d+eAH0wU2bdRj3i3XsafPJHFAArWDsSuNmVR27NAsTLiSlm4nz95Tw1cBQWBnbsC2we5dKA61OKC5blmIkRiUUo9EIuSQwMuyO4EM5bZ23nAM507AL8NXzLpHxy4qUJdEDxnNFI1OO0scMdFSO4k+6WSijnTOgKyt5wsiVTR2ciTGdc5I5Iz+RMbSZgZm45Zj5aAwtllUG+XcM1M0ZvVACKSmYtpWOaSbcqs3EnWLOQrEKawtU27YxK9qWHTs6E4spmmWAIoO18AQaKQQDiH9EKxOscsG1F64vDjUb0jYRCy5F8lYZeRoZwB8j4HTS58uRgdFxzxrDD8lk5FSiM6KYGLt4wXO/0lNauo1LYsKKmAi2ogUkO120M+XIeItNtG2Q/iQ0AX2yUN8j41E74OlwGOnoMlbaiabBqaWAoGzkX4+eHDrOhqULDItOv5wHO4XatATSiUrRvFeP6ObFElhoM5Oigch+6f5h8J+gAniREqQpQsFXQsQipmctEwwpGaIkWjOcM1ANEw4Z5IJpOFOmnUJfkDMdExYAZKiUSoOc2lPwfxdgApHLcViU6Qmwu1F7NCLuCe8wA8adbJzX32zJhI4mQGxMezZVSjvwDT2WAV7qo2mYpTMbGIE7gQFUKABGClY4GtmqLjVvLqNagcOwgHYrE9GFEtgDtrTV3q/JyNBpWpxILEhxNNJ4crtZq2BvBlJcHk2Mx5EDapej8sfIOP4sBsBWyZ7eeCKs13mwUt8j+5hXcWKjr0uXi/FSsRRlPof0tHVnHszYZv1P0BmzJc4MVfuS/HJJm0VzahTO58ywPjfrteY8WMJIISUnFsJA98fk4cPgDv2alTMmzByLmp4toJLh8gQ0yk6iUwZYZ5bIO6vaB4wE9Am+NnYEZHm/WsdkqJpWvd/4qqo1//qquXKtRdPYcO5d1d4qPcPlJURB3iwL/siJRgpGAdCg+GIVspHY+F4XO6BGo5EpmFqSRvL3hI5gVFr+8ZZ1T0bA8zdrq1tLAE5urjXqSxGI/CAsziQnisVcNChKfiNCfZeh8zy6rFImfhwc+QkbwgVvAQFP56bzEEmMRjZfbDT++hIYuQXo4bMafAC40/jrAhgh+Lz5otZ8eRf4B/40RQS6c1ZhM3tD31g4UEz3PKbi1rpUt+ZsQ6If7/IoZUADrT4QIyGhonA3xH0OMhWRQmvIiNEH6CSsFXfWynvqTXCbcHdAU82LabPkaIP0B/QJA2e4NrCrDlk9SQOTCDvgxpDFk4KRFH0qzBqNbF1/tvn01da1heaN+1s3NtDMjSfL03mUdh4lMk5sIWAgeoj5umiuVppP1gNdUFgQ2voJ4dObq1vVDbDS3BoYXUjxQh3BNgya1+f5J4gdxPUnAtuCyXJvZAMUGfeNAgi8WX/Uv3Wr1ni6gZD2eQ1aS3HPAoDNWQUUd5CWYRMRa3S0+1A8Mf5OP4QRRmHWF1S7HR7eR4eHscNcLtq6MBAFWYXJ1HTBcqCvSOM54O2XWmN1vvn9bdCdzfocY/B5APBaNzghrfm40qhQ9OAy8CCyKdhzMZtFclLFQm4WukbBa1EvJ1ybuv3ahCpgAO5uUR8eCFMUNFLJQH0DQ4qCdun8QGH0PMvNecb4Ix+mPhw8OgI2AD4dO3MSgsvIh2dTHxwdGYxIqOp1a6TT0xDsIu1sQgt0DdT4iJDpYiE7TbCXZCCdBwqlSWm1JNtMtadlqhAiulM9guq2jXm4IBylMBLUMhI08JNF6tMoFPMGLkP0QuzIZLGYSaI0kLOAFtQnfeEeYxoEQVqUBAt+jiVZxsSPAeNB425jIqQnT/IHvQg+zJ6B79CrtB9ITUDOIfAD5A0h4dadKtuGnewBERA74hoBvs+zAPz7tgaJfhYrKtjE+ECMSVO/ONo1zuHiRT+7W/sF320Xc74uMAG1x26yACMQvHD8imEeCxl8D5eycG2u15svQU+f327cW9Wa68scZmOED7ZNAx417tU0jNbn57V8+SDEOhiFa42bD5ug7s0vbjeffKuVDve6ZqDaouYyKY44LAfD08C8gtrWx4uUQOAVWZ7TGk8rjU8r7gXKAmAO4jPKUDdW5pA0mbVYrSAddxabFUxoPIQviB5AILaWHmGGYvNxPZSunRWut9W2gHv1NA7t7Ea9eX9u88n3jafrzPasdRGFjoSsVkHaV5caN5/xjw5YNtX0M1+AJxmEeoUyxOwQOCt2pBTPmwZhT8BWhOSE/P3BQhiYcmwLnKoQX69VcATbBItvzZiprGFDfJ6ZTjtsjFxzlUZT0nqfVT4P7dOYplMa252eaaP5hrVJ+PrbmdN9LZyeKBp2Zhs7LiMY/pNCDqXy5VaWYdoONSamy5bGRa8hfPE3agXhTHghA4z2+kdu4iXE7Y6VPu8PEWR3+gTiA7hx2sJ7aRNit9w41MKNTNYndYhRP73bvIcq6wKLo+dGzqSOHsPc33Dq6Ehq+Nz7p4ZGWqXNNWdwz/snzxz758HjIK7CP+aK6fNmxjVJYlJusxYBMaYzlsPRV2rCBHNlKjDj6LnjQyOp9wdPnDk7yEFCixGCgAF4lAFPjMYfiZIG0L3/6ImRwbMixvDdVSqWLcf1qf7GnS3TNvOmPYnKlWIlxnsGTw2e/WDw9LHfp0YGh0NYxbMDkkwykWzqtpbQb2joPpY30FKxL9GaK2vNlVvNyuMfXjSuLDeezwFA0ppXv8VU68sFNG6N+iP4s7VUbdy4reHNXy9jN5tPK/swTv07CUlwIpgAFkhHRHwz7Fxm0LnA8pUF+tHxs3A4tqlEEi2YB5UMHAiZ8Oe3m6tXNIjZAAlrCeLJfBUh390lmCY4Za3xtw3+AnZ9vXlvHTEhJoEf3ML/r71C5wJ8aF6ta+JerXm3AoGY1plI/NNeeXO4hTeTrzElLLtkeIQU60YWDCwhIvwAFwA6MkShVuQIXO8aMB4WeFUwzArMxJuGTqM+6tnO1KRhFVKlkmdS8AJajTTAKQptFaviBHunnajUrJHPkU/Bb9AtLJeVmYZBz5uzoFFZoqFYgOVZnxebtyCiyP7VJW3rywru+m4+fQZOUvC/cfUhOV6wLCDmjae3mlfXfnjR/HIRl00kXrau3sf1ibRImJJioWw/ysVm/S52KJP9nLDn+Fr4avT6H/8RBllZ2/r0j6gFcoNhqpjDXRDM+BMOQfkC6iGo4Iz/HkWiM6GKBNgbw8qVk5dUvkaa1xcRiSg73IQkIgOkC38QFkr330MhntasbDSvVbXewxT1ABr5M0CmymNgb2TgkmgUGZBZpOkCeYjI5pNHvuvuGDFMQ1xrrH7l/lYs4S2XA4NTUNW4tdhcnRfrBXhpGZATjtpYmENoDIs1IABxrohC68Fhj5ZgxwRcNIHxcHGqlWbtodbfC6pIeI2mJa9HBhTXmA7prnqfERL20cl9gBXEPk4cPavcDehkW2Lk/K6ubT5eF9PUDidUgkA6Np94/DRypu2ykzrhPjyGI7oBpQnjAHlYjV2sloAhBUtRk+6su118NG0iomMrCeM0Xla4ETP9IiAyB+A5jPCrZOS3g6ePD53+IOKD9f4eAmRs1hdx0qCdmy82NKAClhMm/AqIAi+ztr75pI50Cfy5LVF0e8vPueIk/ih6UX6YMW0rC6YyQIwI+IXI9ybyZeR94yqxJl9OoU+GmEJZz3yo0GKyRBObfM3v7jeeVDBLAv7RL14UKCidpYszvJTUAd9PZMMPpk2qKWRe/oYrgHNWfR20xwRAOeUUHSOHk8UEJTi42GXFxfFuE5n25uoiWJytpW8b3zxKntdFMcyTOng1BjozMXnxxSuIM9TUsrAzuIcbPa9fuhwLTTT7jXuzch+3k57XgM9o3rn+IDV85tzZY4CiYtJPy6hw6rznpMfDnRtmH6xJrzPKgOKlVJkKgkKG0RXknwxotw5qmgzobGASJEkpc8YsOOVkuFTSqhmTZjK4wPIHuWsMkgEMwXiOrOvW0kMECJRTIkkCr9G48ZUQqEiAEAK7foROHIvJnAIF3ZKNrizsMbtrG1bZ1P7ZnB3E5B5ue0IHHXqH2Ossdwxc6gBYj38cy8lhBRbK1PVVcG83XpIefHxl6+MaFVjdXCcte3ytuQKxN4Em6KqcNguGbRXx1voc7TY+Xt+sLyF3vlwEnzMP3opcOnWssRXSEW6B39ZxAx00nQwauH8xKjcSgxMGraw074CRXZlv3niGmbyb9+MaGvmVNQkPQPRRPnHnv/5fVDlw835zeUHQDXYT+4Ghmnf+HRshqLv+iZgHQuclRNCIgFHjYfTGvWUsHgAH+sXtuLb5fIHy/l/Ut75YEA4Y8aMwhNxeJAV0+Lq+uQGe9k69sfy9hlbwXk3UKrCAyGQCYA1CODcBxGyIhABekGQ9EnMFRhcn0AtDrAyrNiqWSaQ3aNKrt0D8kMbNJx8TZS4/Gg/mmy/pOsEoGOvrawEeMqe3btUE/uEFAFNOq4Q4aNtVanz1CjkOsCeU4wE+r2rkKVbmG59WkGM4JHBDnShAcgGx0a8gu0QdDZBBZNJf5K+PCQjzgUagmeWBiYSvKlGwJN8CET+8EKAfxgY7yvy7oohW81kN0Kkmve1frgE3QVZrlUa96o5HQqAHRKDxdA64hFgLoGy4JATFoCJzSmzMBeoFe0v1K49cmliVQMOYHlxkWr6vCLdgMozYhgsCCAm8osLWcVR6tZ4T3P8kiZKn+rCEzQeAMyFIvOpVzcxJNiO8kctVhdnCUk0WjZxrM36RwsjFTsdoX86daYoH61B42t2nd1CdjWl/6OSRKe9O9byH5T9kKCoyrlhCUEMjL7969yA0ebf03i+Vfe8eLL1HXAAlW7mFKqE1788BEnUnXr+CAThEKKgOwyODv9U6/28vJSGWVgTjNJZ0ogwg2OfrlEq9ugYhm6IbXEtZdqYz4NnjVmm2MIE+iy7H8xlSCRH0fbre+OQ2OiSIT6iK+Ml/oHrCMFrzGiVv2X8rxt8bGNaP9pZconlCLXc2Hq27y8F1bFg/RnC3sULKz9NCrtrGBcAINGuKrZCPZCUeLDQ+u48DP7gl5/nbo2ePnhocGTw7jO1xxZ7S7h+OXVulJI5nL1zhg55oPTE75FK4eh+JRPgeNpvG8zmMNBv122SLYDlweUS+qDlfQ8e1MEe/iR5hsFv31JUvofg3nsyr0wKi38Uiw/dg0u8epE86mi2ISVnc2aqpLcuOWeoUbbEnObuP5z0mwQKu4+LyHT4llXeKRH2dhV7sIcxp3X1gJ5VuG9c2tm5sNJ5fQXZwdwCQWDYTkgyNNxZp1VnHRs4cPwMLvoHwmOz5BuXgMPl2S8xNFIlzMklmNuq3N/9rw8UYLOkwcxfRCckjbNKiVsRWWn1AFrXvWKumc++9m7Pek7zDGE9S/T9ztzUR6ilmXQKyD8+cPI5jS2A2+LtjJ88dH0ydHfyXc4Mw/+MsodvaDl07dvLMsGgWcFUSRrp8fvcg0ngQicWZgLGjvRwUK55GkacBYoKz69vk/IpwQ4LpPDu2DN97zpv7hv+Juf7wAqf2w4uWCYG43yLMd5X3cdY1n+TQdi+BsRa7JgYg5al/TpqD5rlKdavqjJEZFE1RP+A2wRKQQKMBaq7OgfMO6ddDQKIXNIMUY+NkpfmlzjiwVCCSzwm4YMhzIBLZ+gc9WOSVUI3t9yQN9UeNJ0vSYJNEgyG58YxNmMItahygG9NPGAaBzcAuG2vPCZiBrioKdxPCpppwedINob2qfafU7iqLK4T/wRW4RXFFIvhoVudhrSS97MSIAVQxQ3YSczzrDGEQ0ZHdETdfXWvcm1ONDNy/Kk2csLPIccwCr9ZA2WXOnHKHbrL8rkosPf0Bf6ZMIwN/bPz4Hg7IsSNr8bsH4SL9ALNfmBOmwL2KZqC2yl8PYhcHZXcTxcys6DUjlP7SWAdrxljHgDbWgcI/1nFZGAG4M6O0RREP/sIGAPGou56q4mIromGnQVt0bU8UCCDK5qWx6EoByIMiCLsihG3StqOTbrYMH7BeZKmfV4Apqtq4wx8Ui3BQrDRKp9CCVRkoE4gjzXNVSWgs+XHeYOCYQQIVlimByEi7nsyjh1Gkk0SHdix4Jxy03zWjgPOAd588bt64T7nyxkqVvSTorR8woB3RFcQlYSN61lW8h0Thm4eAlzDW3nx5RRKA+I2q8qtyCp4LA/9M4yHQqvxVDa1YpxrP5zG79vmGhhLwwwtaCcozrG1gZf+ddVg9Krjp+OHFWAf9PNZBfPCBXpoMWKU71/E6/lz7TuCr5sY9MVU2wVerWkI6XqHEjysQ0YcbGYCnPKfGn+5L3Acaj0gGiMMFFbiGuFpxTTBb+oeb31VbonjXnvi2GkTXZHvIhKzMI5XgjjAKvb6gUckAs5mHhAUlQmC9FxZh0TS2D4RyZeKCFrW17zvXRd8JylY/uIWTvMt2a6Xqt6IiQFUQeGD51+eadeDPVdw/QZHtFZBdiD8K84cjp07ynhpRQjiKacRKCXSC1f+E7+jKRAD9f4bPnBaukczt/ELjz2uISmA4hZXYsWAH1VwAmRtzBPD/H3pz6XJu/geZ+dWK6nC3A1R/fQX9UFyzuij1VIAaAdDZHT9YkDE6r6eMwikpj2LWfFCR/oAic7Rhd3AsCjg+a5U4Fz+ocbcHIfApk2UqdFEsEmPHMNnt4Mc1y+U8RENKOCq2shgzyAnuhJnJshD6IFXQUchZeMV2qe5CdmBqZZmqcmgQZaUQPkmrs07ajtgY6GaV58+o8k/WkTq4c/PJI40LWIgNLS4Vp+LFoJQ0k7NxHw0i5ovQpCaQfEiUqExVwfkC4JNikKbpogdOB5QPQtuD3Bt8SnGBZnwKOK30J+w4izxxSeGfCuQ88yDSTSSmiCWNgpGbLVvleLo8g8TQE1y2ifVUcXpsVfPPV8h3TeNqYGpDeYIFTkrx0PdCAmieMMnaPMjqNcLGMpHNes76STsrVQF/lHCJFcLFphBsAXELC7gnRGHQvDDPODJ8clVIxEMEzDYIE20P+gRHFZD6+XcKipM5vdYwQoBijpeRLNTnv21AhEtLrC5K4+ErYGPr2gTDVlnUJRhZVeJ3XzDqC8FU3PsUq7OIxZwKlsSR+pAjfI77TyHhge/xN5lHID9HsHIDGmtc3CXNlGtx2NXL+co5uK5fRv201Y0ZUeEtOXfAS6XwROSIRDoA18MvVgPafuI37X+u/0n7sWI36vzHitsYvHv2JtyoVvzPL0q4J+ADrTw9YahqJ/tjNMYQGrlKRrU7DziLxbU+BBMqIR2AR8T0GxkzdrtYNuGHG8gWvo6Iz48kRCpHIETytmifVcBXXcNsEW5jtCTv7rUKj8u0APIgqEKKKOosV24xGpY5JF6BK6x1uCnk46ZX8g3aJbBNo17FmEp0jX3STrG329NqiPm5OVE/jwMCvGTjJbvgyt6Q6dA6MuQGdQZbgEicuC9cJUJb9p7wcQ1UDmlk/+na2EorTYSLiQP0gCvm9tbQjCExX9zG514pkP0TD1fz9jjErgLewSBeWXGRE11YlOnlq+ug+eGpAV+MzKoDJoacP92vySgVDW876n37ot7fHh0eRhAMFt7nfQVKFglQT0Nb87uk33d3zuNWRM6XzJIovJZBaP1PoOyf+m2JkmkHnCASQbx1e1cF4FSSRJbKC3eePoM+KLUNISxaIQ8l7QBWpCXg/CxqvLeB6cXkXqhO1qp+l8w4B+owEbovaDdIicl0NBZDvQ/YDrRWwg3X3JG9UFAmjXXhDOgjFfG4bEKfWpVBpXQRCqf+4UlA3tqofUcFnLx1LuMPygSEpMz9zoYjPY0W91s6ZYa4I2Ec0H33kYCtqm+ExVnbINPuumqJOd29OwyRRTwr0ixe7BaebdB+fiGm6hdOHB06SSiJnAClYp+tYREy+OhlFKvvYdIyPBOwjlAMHeyD+OJxnSYkCk68XVBxgTe1kTDptsUdgm+IHx+wwIltcLeMAviAi4ckqsacN6fAGCgWSwGUwqs2XmJ+jHbyxK4fY2SEhr4OkbMgB1sr8y7KqZGa3q2IWFuqzGpN1rPCQCIe4tRNSDwkGetOe/tkOIRsL6sC9ZEF9Mqa+W43IS6zblilJ/JsP7zg57JBsMEc1SpyZxG0UJghb3pgMsFL+XSLJjbPiuMGrm7p+WLj6/+miJIjjdV5QmLV++BqtEQ8gf86RdklsSYIXMWcZALHFVLAdAg4v34Vah+27t6mdSPwT4ZEaU7z4/1zfshdGc4DIjJ5OOcC9FoFn2q4w/gpaGE05iFzYQk+cbrDbVelc0uIHtTZrngCF+bPr0hOlnFLWmI5ZjbabqUfmk2Q2FOJw2BcV7c+Xk55K9d4OS8ND1VgiQkPFZy+Ho2PMNh2U7FELhybC4mX0LEqHS3rCvX+/FHznti0RQ0HxhCjAc7eWfPxhdI+WzfnQPCE4m6BDM1XvYSComquNeGYy/OcrkTyWSurNfqIzoYIUSjzJylFDmPrbpXCbQ5vaEub0JYvl0Dx1DffbX0urb4AhaiWSjREhcdCu5ZfyaIztH2e2AJdG/fgGpNIyRJZgYGDEZ+3t7BiWfExcxTlr0iCfBU5yHLiNyg6ZQxUIlUyIErAiu/Gja8klX5LRZOEScGk0SKhq+O0t1z7OSJWF50ol8kI4iN2XAQSSEFe1r3CIp43WljyCI8Jx/hquIgXlSqI5SbEkP6ozF9rxH0RZNrYoOII3hvlahhMR8hyGvDxTzDVsPlkwfUq6/xI3prwlv6BgiotyWYfKdwzpa/IKobVoCnxG28B+DIbrfVEPESKHhVddWsq30hV0c+ZUe3KoXbl0HaVQ2q8uJvaoa491Q6FKOSbrCDq+vlXEGFmhdJKdByb1mrbA0U+bldmOW3k8ORRtTd5lKcsEycdDTEYNBRArI+XPe9MW0nKqgIyXFvXeEuUOMeRgwyM3M3hli2l3dUiyfyvl7pGHRTsCEsjepG5MCCUS8TbXVaIXarWDfHXdUl5PA/ycGIPowjEZFUPu/gk2c2KuxkUSmp37ZjUFhNsfFJ3N+x47eeUpeH+/bEl0iZimiAuETOkhDZGWmpttgjq0UG1FGn7oN2eKpdo5m51m1tZvpNrUnetUG48SeOdAQLbFbVDwkKctCYZ/RubOfcRCsXVbauS7bqmPWV4dfjkahP+gCcZ7DntS7uAwdVXsuV7zb9uQ1sW5WYvOdmAtG6vbvtNEO+FfaEUymdwVAQYIi/tYql2sVS7WOoXXyyFgog4+euK8K4gNyLNrPpZNeCUcK0izJvIQ4T7XJFXJD1nc80HXrqPTL2xeqid4oC3tCqqq10V1a6KCq2K8j2is5+6KClIwRDxrauX2nOMyfVMbyy+pO7+92LLYMHUTla3XTbVLptql021g+p2UL2boHr/tVhdobVYO1vmdkXWP7Qi66ee3PyF1Wv9JKPcfZdkuZbKiz7kfbLQCi1vUk3+y3MjBTQUdiqEVlFbwKixXYbVLsNql2G1y7DaZVj/mDKswOEdZK7JsDyrNf9yTdTbUlpEnB4VOOXpGdjDKqa9fLC58dljTFAvYvEWaY7aG1kM0JtldsriFIp1PmOKjzESLkc8p7XK8wBMoKzCa4+II2h4m0t+xfFFqvS11l1xJym3Vv2NHeX0s+VRu+SqXXK1U8lV1x5Krrr3VHIV1MU3WW/V/fOvt9rNOUyMvHELUT5E5jfp662H+u33RCaZ1PVnqFuIfF39kgx+mQoGLUFB4JQymi11T9yLDcWmOe2u8bMkZCPxDpLCHVPPsgTJlwYS27OBMyRFYZJnMzQ3sOf3epA1DRwgKTcM3Byo6Arl+ArtPsnYwDvl/O+sbVJWG0ObihLis+vwVQCt4qHTvG3lN3frAelSDh/kQgAkm8JDYQmpiGA9fEeyXc+029Tr3/nwKMkvadHKPHlKvwWYU2Riz4lN8dTW3p4nFYke3ylzIVbnR3ueVXr07U4UbhcotQuU2gVKb1uBEqhIDeMz4dOVhJ+M3OQTScj+x+uNynzjyzogOu1Da3JK26qtN28uKy5R+kLJXtJ1Ohiao71gks4Lusku6f4TnNXgW53PPgqYtkXVb2n1Une7eqldvbRt9VLX31O99FZWKe0/BqQao/3Hf3T73xf7BcuOtrWV7Zqjds1Ru+aoHfi+pYHv/ouIukOLiHaws+0Kop9EBdFPKYX4S6sa+jmGnvuuKhJK7Ko64ciA+qt7mu4pUGqZmnIKlmrU/UddidcktSuM2hVG7QqjdoVRu8LoH1lhJF6bJ85xXFqgsxtu3N+6VmNHS6+HFyadjhkkuRIHQq0Ezxny1x8h9FtZ8M6c5MQiONvbohZB+sUAFBHZH9oBJ/0Sr7BsXl+A9aVelhcbq/eVyg0RQgu74J+VSKUHZ6XMh14DSsULaO9wU96l3H0pYR1rOsRLfRBhSNcb0NaQyiXmIZ0DmWJ3/6aKl95GBrfLntplTzuVPXXvoeypZ29lT62K/CYrn3p+bpVPKj9AZ32HRLF94CpK9zigqmcl/CZCar3HKSHmnE2RuZjlqmgYkhXZfS1UN2/k+minDQhxsBXnJbGNbxLbZw+VXIOvU6QyIRBiiOxQslzXEgg9bt7Xtq6sEcqS7zAgCfw2OARJrZoqr9H+eBip4jWk0swp2+NsmMLOqKK0MM1dvDgXMDog3z6KwohqtfNdvSm3Nf+LFgQzpaJJFfHL5tNnHJT7OOjWCcjdndc8RrPbsivVe8n4R5VHma/wnUKmqDEjbvGG6a3rzxBF3nWf22aG+B9QfM15Zu3Sq9dnoAOygenWxF7SziEviJbrvJssc8jwnXs6rT84sADS26a523VN7bqmdl3TL7auCUz4PL0XgB+/xNSceGOsf79XKGMw1Z7wgzxvs1r4nPqXim6EemolBy0A59K8yCepHpyjNiFgdwG8zikOrsUz7KfuaQdY/ZaWPvW0S5/apU/blj5177H0KTxOeytroN547Me1TT9a3Efd/7gxX0tp1Q7muF1d1a6ualdXtWPbHyO23X/pUk946dKOdqxdvfTTqF76iWXifmkVTL/EIHPfFU6BPYV52u4PmD41ifNZXZY4texZCicrRFTmreWOq3vqk7hfTlbMqBIQTxcYtwui2gVR7YKodkFUuyDqH/jmO84j8AtuRFoLR7n+Cfi+e65FWZVPC5A5hnCCX3PqP31pES0qd+G9Y1eprL7+GWopswr0icJ49RkEQjK+hwKElSAm417rIxRK4aPXniuLAv6ehl1bb975d0rH8BzctCUMK+suwFeyiskcyp1rcB8JHmvfg/lWxoXUOVENUYp2pSpchfTmDmn6JbOyXdHUrmjaqaKpZw8VTb17q2hqVdk3WdHU+3OraAp7fz3VM7HO0Y3ey9IrG/KF6cEnbQLv0dvmKSqvZ5GW3v5AuPABdlfv1CM33m8tKkYv8PZO9yV11dbggnfMlDpRn8C4rxEQOU2Swt0e5NR4ME87BF5ygxhKf1uPQsbgY209cNYTfFVdtnhWx00hYv6ZJVi8kS4IPLwI3itFUzsXZaluet33DI5wAiS5lZrYn6CImjYFKQYIHjQV9uY974ke5eXpIa+E31+5Ey2aKBPC8wwhbF+u+CGOUjWsvhDSjVTp1bKSOAoc+AIdjsXZEtGbyhI3+OCMOJuhRfc8xFAn2K6B2iFP7DNQ8qB6PL0+aGJed87+9i8BwA0lv6HznnFrVKrqiWfb5pP3ROYeX1XgM4wE38RxAo/2QRexaD/8C3+OGCuZPP8A4EvkPrdJw7crvNoVXu0Kr19shVfw4VnMVYlYB6PWKvLLZ5Fov4CsGaMDikdw15PlhHdtBMxCP/451lbga5h5x4UytbwtjE2x0f1PAtl5LO/+cs7dVaAN++byLUrce6dJAz9RVdbY09OLnkOLQPZX77VD0PGW1nv1tuu92vVe29Z79eyx3issitVfc6DH21gMtovAWLxd78cLimmAn2hA3FIZtoPhbleGtSvD2pVh7Yi/HfHvNeLff91bb3jd245Wul339tOoe/s5pGR/acVwv/x4fN+lcUdPH1dhsqwm0WSZysvVRu2hVxQTQJYCfvpcMwkIri3zevkVrBXyz0Vg7WK3drFbu9itXezWLnb7Xy52G9c7SgYWSzimXe4YuESv6U6dHDo1NJIa7hjoS1zWO2wza9pmIW0eA+DYMdAxRv/BtNcdLNpe3apukI8D3fp4WdloxeNFwYfg1g1mDtY15fV66yg0lGklHyuWTB6S+XwBvoEIcREE1Z3UhFtE6FfFMiq0UARrvR9W1ng5NW8JuSpMggxoX6+QEVi5Fv1waORkTHhQagoBZhVTebQoR4foDuHIqSKcg3XlDLTGN4ABXyqLjvJYd59DEAVpcwr2IugqHOXmi1dYFaC8e7lWQemjZKkH0lBqbnzFbF0EfmpcEYVsFTsgi0xx80GFNnEwYV2N8yKNFaw85mu1KaM8lbMm3O+Ym3W/iLAC/lvKjMHlo8dGhs6cHtaSGgRNZweHR/g0Y9/qREXpg/TzX1ZE3UKM36AujoXmO8MWMqqcqOrGdCA24v6zgycGz4q7wxec2/Fr5XmQNVebaf3HOi6PFc6eOTkoZlLKGU6xWEjlIMQ2bXHT9QUqJvyKe8ubGStt5FLFbNZKyzau2HKbdDGfB4a5Pchp0HC/PTt05uzQyO95RDz5lR7fwvtOQd/TeXqcCr+eLF7Az11wk6pzcGNfAhdhrJDOGeWy9m9F+3w2V7wwaNtFO/qvRm7apI+xgbGCBv9K0Ehtf9acscwL/zJtTpuiRcbMaqmUVbCcVCpaNnNZXUvniunzsgf8h5fjlmPmyzp/zhUnxSdqi/O5rGuj4+JeHA/v+7V2AP75RXNAo7qNCgUIoqrSO+WfrmmIPZc3os3nt3nrAFD6oxj1pdDM+wiCZIyVVYpLtjkDVEliRw90jo+OdaCcj3WMa1bW/UUz8aXcYx0fDJ4eHB4aRqWQfWCfOLN33sFPuCoOndxtF6cLPC7PPhrTte7YZe/GKbhLqFS8PGV09fZFo0TQb0iz4pnpfKkc5U7NQnnaNlNGOW1Z/BpuYCyoXeq8OVum3HAsFgfzCsY1Cp+mzIsZa9IsO9FYYHlgLnGjVDKBNIViHBZpxr/4nTkwoE1djslFQmbOmLaVnU1BHzSvEE6GcihbtLWCjjs6mlWAqUznTdtwzKgiI45hO8lOtUOVtecHtBnq5bwOH6AP6IoFLRrDVTqvFYoOXo/KqXiTUDmO/4rnU/jD/zLv8R8QCnSPShLHtV8lmW0wMf5BCh78IMgMMAT/2aYzbRc0QUjB30Asg687r4W4lR8XP9CpLm15eiJvOUJNylYG/g/0BppN2mDo4Ns0GCzbpzzsHY4O0ZNZT+UbbjFGWhQPwYPFpZ1yAGRYOvAtOnRxnn7UV0wdk9s3VLAs3Kf0Pq3mZRQIHGfraGVQTongsQ4imb4z7WMdRD1eEdOANjwRasUf6WrYif96K/OVf9gV8swxmQSfogPzi7lMqmwB4sFfTxcLNLpZBs9giFtOyMxhxkxbGb6GDS8H5ixtGEzXnDELDpM5fO59MPYs6q/hwsQs33J0CPyLT6PREFiFVnV2ipkiMHjUIrWzULc89sdn0IEI3bNGPe6NCx30OOg6V8Wdq295iLVKJ6qWmYkiBboGKpbMGfmJjKFZA1qU1HzUx0ZwJtJfjuJPYsXH4foBoo25MR7zzZs5roq7kXYs3I4ultA4FW2wV8WciVbLAIOg8gYmzY2l1RFQJ6CrtmGBy/B73iwAIIrtAsWaBA0HtEvc72Xgi284pEQOxlgELSp8l8TuYuixDhVB0qYxYUiRFfQPCass3SKv+KQJtsHK+MmiZlaZZHZ3FIgk2LqyWcQAPzhj7DpEsOS7MPSgpwgfD8Ppv22IpyrFo6rBkXJmIcprHC87tlUCE669q/Xuqn9R8sQgXOtF1mLEDVYujKmu2PBcXHCrINWYBqCQlzuJzrQFbbZ4FNEltfbgcojXCGdQCKamqDPwOgINvz1Z8+d7AYxFXeiLKbPGp9ewlg7AWGybmfvJpMm2rLTaRjUVHmt+lQwg6F3Jgn+qwTfEuAicUooULv0dc/jVNnNoXT9vQq1hwxtbRXo1DibF1t0QfnnBNz2ipmUltlkI5d+vFVGR70+m3OmA1hKaAUAXj7HwaytElOkRYeZ2KdChlErbEOSZS2kLQUlheCl/dlNyjmncgSoOZtWFlz6Hf+eALSCur1vd8JWlt34ArsLsvYic2eaEKmmI0VGJZ7PD1Ks2Zy/MVXsuB41+6J3s/v2+ZdQPj8alrxHwKahs3AWHYOhrWjpz8VNYT+EGxpWWkF53AF3HB48NSZvtA11ML16QQGKsQ3pnvOrBirEOlAqOEHMEz9j90BX65Ido6SkzDcEA8KvcAtMYIsutlQ0+GX19nlL5N72DL0jKvRQTbXK5abQ7i4jOcevi5uo2yLs47SAYVLDaTqiw1chZrfwXa4pKEgbq6AffSh6gNn6xeS+pqWmPEIPZ0m+Sop8jOCUZB2MbXMvxWOv9O4jC4PCxoyePjviEwe1KRd3Dvwe7eYqRt6vujHWBCjXtcsI4bx7D+YqZINc7E4lEPKEKRCoF88mJNIzKbQmgkWa3X867uD1jUPoRXFAyPFFO5IwVfq2Nbj5bbNbuAUDl9CsXz42LBxW1Xnrh01/qlMS8c7v5wC3Qr9DbhnCfWs3o0jHPUfSm31wTQA+wJ4yE4iOCTIgw4U8eZWkUzNNwItGNHE3ED/frrjGl8oonNEB3f/yQjunyRn1J6+w8DAZM1+jGzoS48VD4jb3xHtx5rzVWvvVuShzim/r61PSaf7xeZbwE3rptXMh0dHGXvV3bddkX71IokM17t6egW5llLzfv7nUTgNSWCcRdClR8WLobLzG8EuLxUVwG+D6uw1KUbKvgRGVdiyiCGMBOR11p9tT9o7iMGGOoLkfPjZxJifAndRRUkGJSNMLT+WiI1rtWV+nQbzwUinzWK8ruDx+fQ5RDxRsAM2LeATREczhBZE9/LapN/CeR06aDKPiSL87y1buNFSAqMMto/xTxVP1oKZfolJ/8cB2v4TPPXy7Qlugf1zgP8tCtPhB1Qn4JDoQGO3ePG7EsJlhvAEJA7/3yY8GAyCpyr05j+zEk8sD0vLs9wlWfQUUCDzk0fOzDo2c/ENTnM5L6FgREXW8bDYcRzZrlIpddcOfBA0a642IZpc3fZinTGVt2p8QXupq/dyMirkaQvBGIlmQKxgOJe//kmWP/PHhcOE4UeMOeLKPMEyXSxNuzigX/KC4SE+9gW+EvzItps+T4sSHus6jwSwzmpnalQ8JuRhPjAYSCFzvHVShCl7rH/WgE4uGoGVPdlquapCqkcWbM9TNxR/tNUus+4k1D5bJPqF8jFr59vQFNSjhZNULr1z/RNX/Yg7UQ227yxHwU9uyGwt2IAkfFQpuDBIkggze1gnSpAUeQvN4AedIqeIm013Gv8oqc8sd/xKfiRAkGx7TNymNdUWIkjgOK1/KoxV7srHl43sKNqhZmOnT3IffqhvdYYpCAoBix39s9Ac0v1sl7PLjCK6D4Rd+THsFxDyU8DxQOrAkpk+x/FFfBuc93USOa974c61ghkwXLUcrE3ZMIoiGuMpMddbNqdKJCFIwGodwo/aQERgc0uqLkrMdjcd6j6lQIH4OxO7TfYGNBpJJGV7LLatI8kNgmr9dC03g8a+VyBQPGOAA8jzvFFObbYMIWqNZF3ljBOdEGxqhNrLGZNbgBh7mSUReACzwhw7FxOYEszYArocTmuXYJU3zmTOyyrP3iHUb5kJ/6c5RfHNL4atW9rB3UxKeBeOKfLoO7ExVPvu1Kbk6UUke6f48ShaD59fKAdumjuLKJFuNM79FzxyF8eX/wxJmzgzB3fxOA67Q1eQh3Jr2IkoPrfObwYTXj8Wvf/r/wRu7a9jeezDcfuBOXVQIso0BeVGxIbL58CNBelEHFhKSrNPnlRbzqq/G8QnkfUfTgiwOwLGZpvvFgQW9NY7gpxChCdirZkgY2FrCp29Zt+NIhLYZ421wf5ocp3I2pOr8h6zmoOoqemvjikThyTx4sSGUSQA4IWv2Re6Sgv5yW6z3U+Sj005OJxAOyKUzR0woeOIm1Lrfrja82XNqkiBw9MTJ4NkRCjp05PQKgN3XyzAcCbmCcyE/Zpcy8aU+ahfRsVO5z0OaGb1sDlAujcGggdxaomfziz4+rEWdo1ipwbqNbCc21Gfg0EW5E/PBCPpXH6fNA9ooUnff41KD71CBAy9PHfp+Sc1YSk2STvKxMIEojBMTJGMGD1syLt3sv0g5iK05hsIRYRJSgVQTeTCdKUtFOsXHECQAtA61ZKDcE5l0ULxIS+9owcrpYcIDUlGRFy4KqftHzho2r/0lSHr4Ni6dC/nmt+YCe1Gp8g0+q8nFVdDiSKMjhQjFaC4/jI8BmXhAfYSI7wtuXfpJhVgrncEG8e9PgemhR0XAqrWLc7CPMTKSmC+kpozDJffv5un/G8gDgx0qmqL7xHzAqjhbyHSRKBdSg7aDRoMT+80S9p3S9KGaBCnk6AnVvw1Rp0DHQYRq9Pf1dRldn2sgeymS604cP9xzKdE90dffjn36zK9Pb39fVO9Hf1dvb090/AfirJ5PI9hwy+uGOLHRcLk7bafOEBewDcAB9XjBBAlL5xKEU/4RP3LrthgtGqTxVdPBUNfkUAvCId4h5b1zuhjMndRco6F7OTXe9vC6QhR5w9Hrw4OfCqcShA1z8qyPS1imxg5Ze3yGlo3OiQOfkl66zY4Jvh+Ab/i+hdNylI0bWKfET6NiX8tERQMsuOyGYhD9UCqH7uutGOg/plBLibIy+QzJoO0o7+3CQvgCpPUhql065IdG3khPSxb55oDM/eb1IHnXRG+yiP969qy76kIpenXJIJ4sX9LDc0bbzOozzOszzQvESgnXKNAoA5vC8QyqHZzfKx89hpy2vQ3Afw8cTx6CxevDXnHyMgmt5ZZIxcJAaPf8gfC2VxXsPHPgIcwysSc1b5TLEv0Ag8AF+n0ADb2bYDnUM9OkdxnTGclKArq1MasIEMwJaRQ+ddThGHoBXKmM6ZhqEP3C5VCxbDnXdr3e4xjnlmGUHBw43lrIPnznsGOjUg5ZPNiR7her7Ixirjsty9uDt+czKh1jv68Ow7FNEsf8aH6sGvvzKMnQP2A84/y3WavMOH6IiOtBFPp719TJ2AwjHPfqoqlrHVBn4UsiUOwZA2kBDlF/oxIDyG7Scea5APnbu7NnB0wggTgzCh2ODqcHfDR47h6lD0SiXMqadqaJdnrJKyHmYHcQGi640Y97wkw1XqIn5vsdsRQN+yFzUBlNt9tfL/MJ6ZQn0jgxEeim5zG4iHiv3gdME5q9iWIPx/F16sgJPNsTDCTbEsxNy/VcqBLOapC9YrAyxMPbEKN9Fq7gApVmYICpFdxzsYWcPXMpZE7aBGSoU3sJ0vjQLP3fFu+O92J6qfOkCXEI1S1vnLecAncMJlzvj/fEEss9wSrmiA31x34l4f8dl+Kd3ZMFKmDahbfjpcHdfXyLb2232mF1d/d3ZiUxn/+F0b6InYRjp7gSsb39nZ6anP9HTmTUzRk82k+7De7oyh7oPGV0THZf/PxwMgiY='
EMBEDDED_ASSETS_B64 = 'eNqNOl1z20aSfwVxNgERgwgpiaIEHuLzeu0938ZOKvLePsgqGQSGJCIQ4GJASjqGW/Za2XIib61u177IieSS7+zE3kvqFH8qVd6X/BwC+g/XPTMAAYrM5cEiMNPT3dPf3XD/lEXpKf3UO/26v1Gkzr87XlOv+4FNgiKsDOq+vdlvm0HT8fRSrW5aa83A73q2/iZpNGYb8zXLd/1Af7NcnanPWrWG74V6udLZeLesVSrS2cAxXVW+7Ie+tGR6VPrNR7JK4aFISeA0BvVuGPqe6nidbqiGZCM0A2L2GRLHawFIKED6VjegQKfjO15IgoHmmpt+N+zbDu3Ao94MHLuGf4ohacNKSIrAWLftUb08O9PZkNqO1zY3CiW13AiUGrwVW8RptoDZUqnXGmg0JJ2iZ/b62SuWq7P1uUVxxfWWE5Jax7RtFFF5AZAudjZqHZ86oeN7Og0da22zFvodEJTpOk0PLuk2YN0MwtpkahINA99rpteou761xoSIqiCMSkqyJCHNmVJnI4NAiCePYN2xwxbSequGQi0ybnSXNMJaokqpJJXLGeRlFFIZcNe48nXYlKjvOrb05lx93qxWxEYxMG2nS/Uqgo5kFQag1Q6ozwuz8jrB6bIJJlEEbSKkgZsrOZGTcn1h3kytqjRjzy4mlMUiB8kKsW267nQZ4jVdxyOpxrU5IYYiKquC8rQAHBjqo2Fw6ZVS0cwAhDSLogELErvlSonpgWySeuCv9wVrc4vzi9WFLPEZJE5CsNkiiMdCfFppgbRTklKr3B/BI60xZmcridJA5Ki2UoZhqTWTPc2ZTDW8kIOczUCW5ydAUsvvkOQq89U5u7yY8/hGo1Gx6yOTKaHJzKUmk1hGBjVGAqk00NAzf46vjrxUq4KfJl47Mw+kuOc2zQ73iowGuUt0TI+4OQVmeOfOe9K07SopkXHTXsz6xeKIVt0HC24z8gMtNOsuKVIr8MH2/B4JGq6/rpvd0GdmItQ3V0HeGGx/ZMWu2aFETx6y7pq3nEHYUkO7P4Fti8yRhRPOnTBdFbGJ3ZrZHdE9fz0wO4Ax724LjVJjdnIQG4QY/KUw0L2wVbRajmsXSI94Sg5Do9owG3WQh++7dTNItdxwyQbTFiqLh0Ngpk11i2AEr+F+EVnS8U9qMDPcYAS2JLyNWeEERdbnrarVGFNkJaPICpfJCDVLOv3s9sLE6CcwCyXNsXDRJmHgWD8r+zQCiRlFIoo0t3JbGtEhM2SRWDkdlrL2MDsiLNVTi+MiNL3NdUiXJGsRARognOiGxO6PRSTh45V6tVy1wHVcP+xnrFAYL+M7H1YzXldmknD9Zj9rZp2AMJZqUxgci8sZV5lFV6nlPGmgeX5IspxlEzfLVgFhyOAUWK7p/lzLmMtYBo99oRl2aRL8ZiqV6lyVM7vOydV91x5oDR8kNXaHJGAuVK2F6kkJUbNH/r+qQnBd+gn7xdzPYsI/t4ntmIVRLlpAwSn9NIdlSxQReqfaZlpITYFgMT5XO2VKpZQQOs/8pLIklyqquWyUy3ogxsHgqnfV0xwPSqKuhcGIZgV9Mi8uVKbY2CCHRGrNJfXrzBxLoEwsORAen8d9LQdi+Tbp5+NpAGXHiSpkrNCYzmMqxRRgU9j8BNZ4suGq0hvOBrHHwVrq2II9gdtpzFz1TqmnPsY2oFBQjPf6cpcSLE0dK5TBTgGvZBnrjmf769rvzr9fKq+e++DyhYu/xspGMoFijxgltWM28afhuBDfDTk5iS5MjbNBYG4WKooG225BlhWxGzpgXU1qLK+IBeIahdBsqpZLWUeA/PCdDcP2rW4brEezoE0IyXmX4BuCKzWnUYAjyoZmuSall802MeAdlxHLG4YBvkcaoB0bYHDpHLdEYwnu6TUZlALhJOwGnrQxENyYnc6IapOEguQvNy/aBRk203tw3RgE7mY7PVmV+UK6D5pmm/ALm4nyYRv+apSEZ0NgA7yGAFqskF2zTlyAjP73KP50V4q2Hw+fbQlwoEs8yMWAjvcPsup1XVe1tLZDKahfSai2TcdjZPEBsAnvAzycvQQVYFURRKnBQrLIQRJU3KOZoq56ja7H7EwCi/Bc30QMbaL2TLdLFKEuc5q6ZBPom1orIA3jtx+9L3Y/qH9MrBDeCx5Zl37p+vXC8r8ufXBZo0w/TmOzwNDzq84oK2o/3OwQHbXgQuhHdt79mPqePFAQfcKZgZzBuwVAawWlBrK+4rQJ3IzZOjIQkJ6/lmGA86aokHFKyiBzW9ry16+gKxZ4p6OKOKlCC0CTe6NrZewgWyeOrB4XGRB7StdbBBhmy/gkK2oY8NeAgTBikIOC86bVKqwZ74VB1hjCljCENQVEgBiS3RAqZ0YpWcDNVLNQ5XEq+ASEgECBbwWS35Cyl4PnEUMjuDWEE/wloD0jWF5bqcGJHI+24LFnGAY+fPIJPqXOeQYt/vje3Xj/SNZRwYAYAWSfqUc+M24Tit5TFDamSMgARWWQuy3uKjVUTCoQ3FZqXI3JIgLk9M2rrUTZa2pvXAxcxRwMBDJ2Vwi9XqoRFVfqyeWVcdqM6Qxp2INCoJAQpIaFJUqT0GUeb1dqwiFTcyjUVQeDJUQ8BwTGwZT6pOAi2m8RiIBx4kLAr4MjtCFDTINWBkoNgwSAQZVgkXPYEcBOQSxnbs7lIppjWb32i76ldQIfNXjRHkg/vpJwRUSr9L3ldwM6iLf3hodbuBZvP4wP9qQPN8MWSATejv/zM2n49HD47PU1Ls9WOQ19oROCRtlqJ12k0GzAHXxlGous5YXf4YudeO9B/OUT6fjuk3jreym+vxP9923GxTdbEH2l+C787LKFw6PhswMp+vvj6M9Hgi1gCU7eiz9/AQfj/V0JVG+bVHAbPXotDV98G73cAuzR9q1o+6GGC/GDQ4C4Hn+9H+9fl+IXe/F/fSp9eHZpSTq+vRMfgBM853TviUNyGtmxXsuYIL5CuMAuMF1VVNYCjN7FUdYos1VKmLHBebYmo1HCr2aboQlmo3EbWgIbSbIkXzldTiAzAm3NCLFfW7py/kNQ6PgR6NDtJRyGFWZUuQTmJAEQ5YobCIUKh8KehZkNBSPx4I0OQHhSdHgIUmE6uLnPtPTFc4YDojk0c+dYAGLmFH+1g8aCe2CB56DwDgfDwz0pevY8fvD8msLqBaq5xAw8EvxL2HYTT8tUUWMyytZXIKrMK5ReiOXKpfeNHM68kDIHlAEnxmslYhuMTarx90JgvPcGf/zkEyHFYFnmd7r4K3lFAYKW27UJLXCw1CywuaXZ7MObXbAFaHfZOmt7GfsdjblxCxobrNcS9Mwib34mM4ifqEzG4TlKlqMNzhVb8D2rZXpQGbLKUhSHKWSNF421JNwldRe0kbxe4lFOWJYMDhEfbElAF27kQcU2AQb8JN6/zWFqiEeDFhbDvW0wYgaQw6Oj5QKug4G+M1t6z0hUAnr0mmGLY4BLYPnA74DQxWKGZYbtBMjp0xkQppeRIXRURMtuoI4yRcbuEQGwc7o8+MMv+pfMsIV+UMgwqo7xCd70Lpj72OoApHBNGXNWxgsUQpliBnZVqiXVTIqEwo1IQfCiZqgrE/y/w9IJdtGpcfz4Cn6jZ0fDw7/++Cq6tRUf3BARkXluLgJK8b0tiLgY4TCKJZgZmcS6KSQIi0wJXXwzF5FmU7v5akcE9Oj2daQpxU//R5pnNpKVBMeBiYM9nBMVF5dDSV1U0q2P0F2T9fkxpjmIkmmApvCMW5M5Hj6/Hh9ySe1DutmShkeH0YP9UQJIvpbwcky8AGpECZiTlTEf3zo4vrl/fHdPGr56Pfz+UGXp6/Zj0E5SdUnR0zvx3pYqJX7EVpGRaPt5tP1DvJ/wcnx3N946ir+8o8kjctz/Wc+XVirppu+x6MN8JAdi5M/n5JJspUZg9kjO7WVcwRzOuJKwQJTig+vx/UdoFbCX8820Y7nGO9lsIbIaEMsPbA3biGtqP61Y9Ez1ogpgPT2mcnVfgDBNgg5E6xD2GqM3lV1V7TqXiEm7AWuCqC4aX7VNTA9HJHJaPkhc90LKEPKiJ4dgBbtovuA9kAF/fAX1CkCJMkHUJtD45ASHV8/bJe6mYuy222awOc2Z+O4U20yLpfjmjeN7T4ALtEtRLYujqny8uxXf/zZpXTGqJaUAxqrKtZMnRm6Knpn3NRHmTpwZxRNZzRcCE/Bz5Lx4kFk6iZ7eAhPfHT79R/zFTry1l1R38T/uRH/bE9WdPAGVAONFafz8lqxe7rbrEO0tHNKaUCQ3IIZ6FlkF6fqeTRUt9C/g6KYwq4ymBpBLqdE3Pb9tupurFhYqOrjpa8YKZLubN4DG3vEXu1J8axdEAg08dJIpINSIACGCWhawGZg2WUUqDtT4nrWpy9Hfvh8eAa/7r9HAUAABcU3UyGrDDMD2bV6c6PKFsx9Jw8Od+OXd6NEBgDl0DV3DdF0gubfFiNz/FvW791hW2Twq3QerjR9CMXsCrNuBspKsor2vtqkui7J+ePh9vH0gHf/ls+GrA6lNR4DmxgQ4KJLhtgyOZ1cGI7QxfHUUfQ2B6uB29Oc70Tc3GBgbXAODicgOH4NFRjsg3i/vxE+/4+Iyu7YTrkLwcezVOoGWioC8nnGpQsERPdgD3CJTyWpotjtA2SYhOA6xU1Dcf7mX7qdfVVJUe1vxD0CtUV5tQjuy2umArMtwtT0IzNJbkEEtaBscdn3B7vHNHSiupOOvtqB0TWygRwJIxXCXb66j+x/ffQwbiS1GP3B5Wy1iraW3hq7kKZw/2AEe4cB30d+fcGwdk1K8whjEn14wr0IIMI7U0iBlY3MGBiKnpRoJ8JOOgWObJRIWluUTtgeuNtnUcGNkXPCWtSU2r+JXxR2oYleyM4flNZW5zwrOHvjwRgM+AgeqYu5XymgUwZ0S5xFQ9+cGDnz+gIFNQeeCboMkERLKAs8Q19NaJi2sKWcKwst7yjvlUmnk02XltPxWfmLhMUj57bff4Gc0h170QtJkp8/00qMzSu5YHcoz8BH5TKF3Ro53b8kgfIzvt+Ldh7Ki98ZDEburyphVBhPidmdUWIrQhO3dKut8IRCxHjFNGfx4miMQZlqGwL0p+YEXdLyQiO9dl6I/PsYGV4x10iZ9jK22bxN31eyCuwe05XSU3ISJbKKel2X8/x9gYMjNqgs5FKcYzG/5UtLQi7SKLRBLtKtdj4EmFpUCcs9nVdOKgkNkYSlkc0U5cccTTHO4CdLIQMq/vSiSIMYSSDYSH3Gwnp+POI5vPI++/g69XIqPHoCvQW2FWT9T0kB6ElVNZoaQS1Y4dohf3okeHEB9IKIlI8SLt4fRI4hgh3+N73+amSbk1c9sopZKQWMxhL79du41ScX9pBClIZ1iJGxvSkmeJFCesyDzb++dKHKhS2KY4RdH6j7oc2z+l+MsmQ2PEYbTYxdl28oApybptBvqJD4wyc+KEEaIBD+Tjt+erU0bpjD4n749huyk8mBlCS/OMY/JOQeggcWGrJyLdFrRbvKuvo2mjlvYh7ebGoAb8I89m25oJNVYnjBYTDLv2j443oXif3cn+vyOBCvR548SPvIXAYzKICshzpCYaICOpkgDdibLIs8ZZBhYSJMZS7xJyBhZgVA7DW38RJHXGACkEQa/10LpeelXEuTK6NsjNmfM1OaDzJwRXCt69h/QCr28jiUGeCFLuV/8CdogbKJzH8D+7eL5360uXTl75bzRP9kRjAZnelLzqskcSh+NpNRsuarnq1coqwIaYumri+FQ0m3OKlN6iUENJ8hgMfhZzjFKNeefKjXn9OnEWOrZtkmp1dM5H81M+Bwc7tWnjLHHRnvOpKmemOSxz6GjQSyfXTsrYk6LJHITExFLMN028FslFM6a568XlJr4vugko6LR98VkvAIPv++CR5/1nDbLAhcCs03Y553pO30hNK3Tpa1CH0Wgw32Ad/w/OfYlqp9gpRiqLELqMmO8GPrFcN0vmgnyYgOxU+zDptiKlleckShOUQbpPJ/xU8996asrg/SuAwX+XPVODf4PT6yS7w=='

def _decode_b64(payload):
    return json.loads(zlib.decompress(base64.b64decode(payload, validate=True)))

def load_embedded_bundle():
    global SOURCE_BUNDLE, DASHBOARD_ASSETS, REFERENCE_CACHE, REFERENCE_EXECUTION_SERIAL
    bundle = _decode_b64(EMBEDDED_BUNDLE_B64)
    loaded_assets = _decode_b64(EMBEDDED_ASSETS_B64)
    if bundle.get('projectId') != PROJECT_ID or bundle.get('version') != 'wel01-integrated-2':
        raise ValueError('프로젝트·버전 불일치')
    if bundle.get('fingerprint') != _sha({k: v for k, v in bundle.items() if k != 'fingerprint'}):
        raise ValueError('원자료 지문 불일치')
    if bundle.get('assetFingerprint') != _sha(loaded_assets):
        raise ValueError('화면 소스 지문 불일치')
    for m_id, item in bundle['missions'].items():
        if len(item['stages']) != 5:
            raise ValueError(m_id + ' 5단계 누락')
        if sum(s['minutes'] for s in item['stages']) != item['minutes']:
            raise ValueError(m_id + ' 시간 불일치')
        if hashlib.sha256(item['referenceCode'].encode()).hexdigest() != item['referenceCodeSha256']:
            raise ValueError(m_id + ' 참고 코드 지문 불일치')
    namespace = {}
    exec(compile(bundle['runtimeSource'], 'WEL01_reference_runtime.py', 'exec'), namespace)
    bundle['_reference_runner'] = namespace['_reference_table']
    SOURCE_BUNDLE, DASHBOARD_ASSETS, REFERENCE_CACHE = bundle, loaded_assets, {}
    REFERENCE_EXECUTION_SERIAL = 0
    print(f'WEL01 M07 내장 교육 자료 연결 완료 (5단계 · {bundle["missions"]["M07"]["minutes"]}분)')
    return bundle

def _connected():
    if SOURCE_BUNDLE is None:raise RuntimeError('먼저 첫 실행 셀을 실행해 내장 교육 자료를 로드하세요.')

def _parameter_code(mid,code,parameters):
    allowed=SOURCE_BUNDLE['missions'][mid]['parameters']
    if set(parameters)-set(allowed):raise ValueError('정의되지 않은 변경 파라미터')
    if mid=='M02':
        value=parameters['FLATLINE_RUN']
        if not isinstance(value,int) or value<1:raise ValueError('고착 분은 양의 정수')
        code=code.replace('def flatline(series, run=8):',f'def flatline(series, run={value}):')
    elif mid=='M05':
        value=float(parameters['SCORE_THRESHOLD'])
        if not 0<=value<=1:raise ValueError('문턱은 0~1')
        code=code.replace('eng = (test["score"] >= 0.5).values',f'eng = (test["score"] >= {value!r}).values')
    elif mid=='M09':
        value=float(parameters['MIN_RECALL'])
        if not 0<=value<=1:raise ValueError('재현율 하한은 0~1')
        code=code.replace('(g["재현율"] >= 0.90)',f'(g["재현율"] >= {value!r})')
    inserts={}
    for node in ast.parse(code).body:
        if isinstance(node,ast.Assign):
            names=[]
            for target in node.targets:
                names.extend(x.id for x in ast.walk(target) if isinstance(x,ast.Name))
            updates=[f'{key} = {parameters[key]!r}' for key in names if key in parameters and key not in ['MIN_RECALL','SCORE_THRESHOLD','FLATLINE_RUN']]
            if updates:inserts[node.end_lineno]='\n'.join(updates)
    lines=code.splitlines()
    for pos,value in sorted(inserts.items(),reverse=True):lines.insert(pos,value)
    return '\n'.join(lines)+'\n'

def source_table(mid,parameters=None,refresh=False):
    global REFERENCE_EXECUTION_SERIAL
    _connected();item=SOURCE_BUNDLE['missions'][mid]
    p=dict(item['parameters']);p.update(parameters or {})
    key=(mid,_sha(p))
    if refresh or key not in REFERENCE_CACHE:
        tuned=None
        if mid=='M10':
            # 현재 M9 실행 결과를 우선 사용  아직 실행하지 않았으면 기본 M9를 실행한다.
            matches=[(k,v) for k,v in REFERENCE_CACHE.items() if k[0]=='M09']
            if not matches:
                source_table('M09');matches=[(k,v) for k,v in REFERENCE_CACHE.items() if k[0]=='M09']
            m9=max(matches,key=lambda x:x[1][1]['executionOrder'])[1][1]['selected']
            tuned={'heat_temp':float(m9['th_heat']),'cold_temp':float(m9['th_cold']),
                'min_persist':int(m9['persist']),'wind_corr':float(m9['corr'])}
        code=_parameter_code(mid,item['referenceCode'],p)
        raw,meta=SOURCE_BUNDLE['_reference_runner'](mid,code,tuned)
        REFERENCE_EXECUTION_SERIAL+=1
        meta['executionOrder']=REFERENCE_EXECUTION_SERIAL
        meta.update(parameters=p,md_fingerprint=SOURCE_BUNDLE['fingerprint'],reference_name=mid+' 교육용 참고 구현')
        REFERENCE_CACHE[key]=(raw,meta)
        if mid=='M09':
            for stale in [k for k in REFERENCE_CACHE if k[0]=='M10']:del REFERENCE_CACHE[stale]
    raw,meta=REFERENCE_CACHE[key]
    out=raw.copy(deep=True);out.attrs['reference_meta']=copy.deepcopy(meta)
    return out

def TODO(mission,step,instruction):
    raise NotImplementedError(f'{mission} STEP {step}: TODO를 실제 pandas 코드로 완성하세요. {instruction}')

def _rows(df):
    return json.loads(df.to_json(orient='records',force_ascii=False,date_format='iso'))

def finish_mission(mid,raw,stages,stage_specs):
    _connected();canonical=SOURCE_BUNDLE['missions'][mid]['stages']
    if stage_specs!=canonical:raise ValueError(mid+' 내장 기준과 노트북 STEP 정의 불일치')
    if len(stages)!=5 or not isinstance(raw,pd.DataFrame) or raw.empty:raise ValueError('5단계와 원자료 필요')
    expected=list(raw.columns);original_ids=raw['원행ID'].tolist();report=[];cumulative=[]
    for k,(df,spec) in enumerate(zip(stages,stage_specs),1):
        if not isinstance(df,pd.DataFrame):raise TypeError(f'{mid} STEP{k}: DataFrame 필요')
        col=spec['outputColumn'];cumulative.append(col)
        if df['원행ID'].tolist()!=original_ids or not df.index.equals(raw.index):
            raise ValueError(f'{mid} STEP{k}: 행 누락·재정렬·중복이 발생했습니다.')
        if any(x not in df.columns for x in expected+cumulative):raise ValueError(f'{mid} STEP{k}: 필수 열 누락')
        if not df[expected].equals(raw[expected]):raise ValueError(f'{mid} STEP{k}: 원자료를 변경했습니다.')
        if df.columns.duplicated().any():raise ValueError('열 이름 중복')
        preview=df[['원행ID']+cumulative]
        report.append(dict(step=k,title=spec['title'],goal=spec['goal'],outputColumn=col,
            learnerHtml=spec.get('learnerHtml',''),rowCount=len(df),columns=list(preview.columns),rows=_rows(preview),minutes=spec['minutes']))
    location=OUTDIR/mid;location.mkdir(parents=True,exist_ok=True);stem=f'WEL01_{mid}'
    stages[-1].to_csv(location/(stem+'_analysis.csv'),index=False,encoding='utf-8-sig')
    meta=copy.deepcopy(raw.attrs['reference_meta']);plots=meta.pop('plots',[])
    details=dict(projectId='WEL01',mission=mid,originalRows=len(raw),codeSteps=report,
        referenceMeta=meta,mdFingerprint=SOURCE_BUNDLE['fingerprint'],
        structuralValidation='원행과 원본값·누적 열 검증 완료  코드의 의미·교관 PASS 판정과 별개')
    (location/(stem+'_code_report.json')).write_text(json.dumps(details,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
    config=dict(projectId='WEL01',mission=mid,title=SOURCE_BUNDLE['missions'][mid]['title'],
        scenario=SOURCE_BUNDLE['missions'][mid]['scenario'],hours=SOURCE_BUNDLE['missions'][mid]['minutes']/60,
        stages=report,sourceColumns=list(raw.columns),sourceRows=_rows(raw),meta=meta,plots=plots,
        fingerprint=SOURCE_BUNDLE['fingerprint'],role=ROLE)
    safe=json.dumps(config,ensure_ascii=False,default=str).replace('<','\\u003c')
    document='<!doctype html><html lang="ko"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>'+stem+'</title><style>'+DASHBOARD_ASSETS['css']+'</style></head><body><div id="app"></div><script>window.WEL01_CONFIG='+safe+';</script><script>'+DASHBOARD_ASSETS['js']+'</script></body></html>'
    path=location/(stem+'_result.html');path.write_text(document,encoding='utf-8')
    if not globals().get('HEADLESS_VALIDATION',False):
        display(HTML('<iframe title="'+stem+'" sandbox="allow-scripts allow-downloads" style="width:100%;height:1150px;border:0" srcdoc="'+html.escape(document,quote=True)+'"></iframe>'))
    print('HTML:',path);print('행 보존·누적 열 검사 완료  최종 PASS는 교관 판정')
    return path

load_embedded_bundle()

## M07 검토·승인 상태와 담당 책임 확인

가상 건강지원 절차에 검토 대기, 보류, 임무 조정 승인 대기 상태가 섞여 있습니다. 현재 남은 결정을 찾고 다음 담당자가 무엇을 계속 확인해야 하는지 기록하세요. 권한 우회 차단과 감사 기록 시험, 긴급 연락 경로의 모의 시험은 서로 다른 증거로 확인합니다.

실습 시간 180분. 준비 셀 → STEP별 설명과 코드 → 마무리 셀 순서로 실행합니다.


In [ ]:
MID = 'M07'
PARAMETERS = {'HOLD_LIMIT_S': 60}
raw = source_table(MID, parameters=PARAMETERS, refresh=True)
print("입력 표의 행·열 수:", raw.shape)
display(raw.head(6))


### M07 STEP 01 — 아직 끝나지 않은 검토 건 표시하기

<h4>이 단계에서 해결할 일</h4><p>검토 요청을 접수한 것과 결정을 마친 것은 다릅니다. 현재 상태표에서 대기·보류·임무 조정 승인 대기를 찾아 다음 담당자가 계속 확인할 수 있도록 표시합니다.</p><h4>시작 전 준비</h4><p>각 미션의 STEP 1~5는 앞 단계 결과를 이어서 사용합니다. WEL01_m07_student.ipynb의 첫 실행(환경 준비) 셀과 이 미션의 준비 셀을 순서대로 실행하면 현재 참고 프로그램의 결과가 raw라는 입력 표로 만들어집니다. PARAMETERS는 기본값을 유지합니다. 이 단계만 다시 실행하려면 그 미션의 준비 셀부터 바로 앞 STEP까지 순서대로 실행해 둡니다.</p><p>이번 입력 표는 <code>raw</code>, 완성한 결과 표는 <code>step1</code>입니다. 새로 만들 열은 <code>검토_대기</code>이며 배정 시간은 36분입니다. 노트북의 <code>M07 STEP 01</code> 코드 셀에서 TODO 호출을 본인이 작성한 pandas 계산으로 바꾸세요. 결과는 DataFrame이어야 합니다.</p><h4>입력값의 뜻</h4><ul><li><code>status</code> — PENDING은 검토 대기, HOLD는 보류, EXCLUDE_REQUESTED는 임무 조정 승인 대기, CLOSED는 절차상 종결 상태입니다.</li></ul><h4>수행 순서</h4><ol><li>raw의 6개 검토 건에서 status를 읽습니다.</li><li>PENDING·HOLD·EXCLUDE_REQUESTED 중 하나이면 검토_대기=True로 표시합니다.</li><li>그 밖의 정의된 종결 상태는 False로 하고 step1에 저장합니다.</li><li>미완료 상태별 건수를 확인하고 대상 원행ID·현재 상태·남은 결정을 기록합니다.</li></ol><h4>결과를 읽는 방법</h4><p>이 실습의 검토_대기는 미완료 상태 세 가지를 묶은 열입니다. 단순 접수 대기만 뜻하지 않습니다.</p><h4>직접 확인할 가상 예시</h4><p>다음은 규칙을 이해하기 위한 가상 사례입니다. 실제 실행값을 대신 제출하는 자료가 아닙니다.</p><table><thead><tr><th>가정한 입력</th><th>기대 결과</th><th>이유</th></tr></thead><tbody><tr><td><code>{"status": "HOLD"}</code></td><td><code>True</code></td><td>보류도 미완료입니다.</td></tr><tr><td><code>{"status": "EXCLUDE_REQUESTED"}</code></td><td><code>True</code></td><td>아직 승인되지 않았습니다.</td></tr><tr><td><code>{"status": "CLOSED"}</code></td><td><code>False</code></td><td>절차상 종결로 분류합니다.</td></tr></tbody></table><h4>완료 전 확인과 기록</h4><p>원행ID는 계산 전후 같은 자료를 찾는 번호입니다. 입력의 행 수·행 순서·인덱스·원래 열의 값을 유지하고, 앞 단계에서 만든 열도 모두 남깁니다. 각 행의 계산 결과는 새 열에 넣습니다. 실제 불리언 True·False와 문자열 "True"·"False"는 다릅니다. 값이 없다는 뜻의 결측값을 False나 0으로 대신 채우지 않습니다.</p><p>계산 뒤 해당 출력 열과 입력 열을 원행ID로 함께 확인합니다. 실제 자료에서 해당 사례가 있으면 최소 한 행의 입력·출력·판단 이유를 기록하고, 해당 사례가 없으면 0건 집계와 아래 가상 예시 시험 결과를 남깁니다. 관찰 내용은 5단계 완료 후 HTML의 해당 STEP 기록란에 옮기고 기록 JSON 저장을 눌러 보관합니다. HTML 입력란의 글은 페이지를 닫기 전에 저장해야 합니다.</p><h4>주의할 점</h4><p>CLOSED라는 상태만으로 실제 현장 조치나 진료가 이행되었다고 쓰지 않습니다. 정의되지 않은 상태는 임의로 종결로 바꾸지 않습니다.</p>


> **교수자·해설용 FAIL 점검 포인트**: PENDING만 포함하거나 승인 요청을 승인 완료로 처리하거나 CLOSED를 진료 이행 증거로 쓰면 FAIL입니다.

In [ ]:
# M07 STEP 01 — 아직 끝나지 않은 검토 건 표시하기 (교수자·해설용 예시 풀이)
# 해설: PENDING, HOLD, EXCLUDE_REQUESTED는 미완료(True), CLOSED는 절차상 종결(False)입니다.
step1 = raw.copy()
allowed = {"PENDING", "HOLD", "EXCLUDE_REQUESTED", "CLOSED"}
unknown = set(step1["status"].dropna()) - allowed
if unknown or step1["status"].isna().any():
    raise ValueError(f"정의되지 않은 status 값: {unknown}")
step1["검토_대기"] = step1["status"].isin(["PENDING", "HOLD", "EXCLUDE_REQUESTED"]).astype(bool)
display(step1[["원행ID", "검토_대기"]].head(10))

### M07 STEP 02 — 보류 중인 건이 상급 검토로 넘어갔는지 확인하기

<h4>이 단계에서 해결할 일</h4><p>보류가 길어진 건은 다음 검토 주체에게 요청이 전달되었는지 확인해야 합니다. 보류 상태라는 사실과 상급 검토 요청 기록을 함께 읽습니다.</p><h4>시작 전 준비</h4><p>각 미션의 STEP 1~5는 앞 단계 결과를 이어서 사용합니다. WEL01_m07_student.ipynb의 첫 실행(환경 준비) 셀과 이 미션의 준비 셀을 순서대로 실행하면 현재 참고 프로그램의 결과가 raw라는 입력 표로 만들어집니다. PARAMETERS는 기본값을 유지합니다. 이 단계만 다시 실행하려면 그 미션의 준비 셀부터 바로 앞 STEP까지 순서대로 실행해 둡니다.</p><p>이번 입력 표는 <code>step1</code>, 완성한 결과 표는 <code>step2</code>입니다. 새로 만들 열은 <code>보류_상신확인</code>이며 배정 시간은 36분입니다. 노트북의 <code>M07 STEP 02</code> 코드 셀에서 TODO 호출을 본인이 작성한 pandas 계산으로 바꾸세요. 결과는 DataFrame이어야 합니다.</p><h4>입력값의 뜻</h4><ul><li><code>status</code> — HOLD인 행이 보류 중인 건입니다.</li><li><code>escalated</code> — 참고 절차에서 상급 검토 요청이 발생했으면 True입니다. 의무 판단이 완료되었다는 뜻은 아닙니다.</li></ul><h4>수행 순서</h4><ol><li>step1에서 status가 HOLD인지 확인합니다.</li><li>같은 행에서 escalated가 실제 불리언 True인지 확인합니다.</li><li>두 조건을 모두 만족하면 보류_상신확인=True로 만들고 step2에 저장합니다.</li><li>HOLD인데 False인 행은 요청이 확인되지 않은 건으로 기록합니다. True인 행도 담당 역할과 다음 확인 시점을 남깁니다.</li></ol><h4>결과를 읽는 방법</h4><p>True는 보류 건의 상급 검토 요청 기록이 있다는 뜻입니다. False는 비보류 건일 수도 있으므로 상태와 함께 읽어야 합니다.</p><h4>직접 확인할 가상 예시</h4><p>다음은 규칙을 이해하기 위한 가상 사례입니다. 실제 실행값을 대신 제출하는 자료가 아닙니다.</p><table><thead><tr><th>가정한 입력</th><th>기대 결과</th><th>이유</th></tr></thead><tbody><tr><td><code>{"status": "HOLD", "escalated": true}</code></td><td><code>True</code></td><td>보류와 요청 기록이 모두 있습니다.</td></tr><tr><td><code>{"status": "HOLD", "escalated": false}</code></td><td><code>False</code></td><td>보류 건의 요청이 확인되지 않습니다.</td></tr><tr><td><code>{"status": "CLOSED", "escalated": true}</code></td><td><code>False</code></td><td>현재 보류 상태가 아닙니다.</td></tr></tbody></table><h4>완료 전 확인과 기록</h4><p>원행ID는 계산 전후 같은 자료를 찾는 번호입니다. 입력의 행 수·행 순서·인덱스·원래 열의 값을 유지하고, 앞 단계에서 만든 열도 모두 남깁니다. 각 행의 계산 결과는 새 열에 넣습니다. 실제 불리언 True·False와 문자열 "True"·"False"는 다릅니다. 값이 없다는 뜻의 결측값을 False나 0으로 대신 채우지 않습니다.</p><p>계산 뒤 해당 출력 열과 입력 열을 원행ID로 함께 확인합니다. 실제 자료에서 해당 사례가 있으면 최소 한 행의 입력·출력·판단 이유를 기록하고, 해당 사례가 없으면 0건 집계와 아래 가상 예시 시험 결과를 남깁니다. 관찰 내용은 5단계 완료 후 HTML의 해당 STEP 기록란에 옮기고 기록 JSON 저장을 눌러 보관합니다. HTML 입력란의 글은 페이지를 닫기 전에 저장해야 합니다.</p><h4>주의할 점</h4><p>열 이름의 상신은 상급 검토 주체에게 판단을 요청했다는 뜻입니다. False 전체를 요청 누락으로 세지 않습니다.</p>


> **교수자·해설용 FAIL 점검 포인트**: escalated만으로 판정하거나 HOLD=False인 행까지 모두 보류 요청 누락으로 보고하면 FAIL입니다.

In [ ]:
# M07 STEP 02 — 보류 중인 건이 상급 검토로 넘어갔는지 확인하기 (교수자·해설용 예시 풀이)
# 해설: status == 'HOLD' 이고 escalated가 실제 불리언 True인 행만 True입니다.
step2 = step1.copy()
step2["보류_상신확인"] = (
    (step2["status"] == "HOLD") & step2["escalated"].map(lambda v: v is True or v is np.bool_(True))
).astype(bool)
display(step2[["원행ID", "보류_상신확인"]].head(10))

### M07 STEP 03 — 임무 조정 요청과 최종 승인을 구분하기

<h4>이 단계에서 해결할 일</h4><p>군의관의 요청이 등록되어도 최종 승인 전이라면 결정이 끝난 것으로 표시할 수 없습니다. 임무 조정 승인 대기 상태를 따로 찾아 남깁니다.</p><h4>시작 전 준비</h4><p>각 미션의 STEP 1~5는 앞 단계 결과를 이어서 사용합니다. WEL01_m07_student.ipynb의 첫 실행(환경 준비) 셀과 이 미션의 준비 셀을 순서대로 실행하면 현재 참고 프로그램의 결과가 raw라는 입력 표로 만들어집니다. PARAMETERS는 기본값을 유지합니다. 이 단계만 다시 실행하려면 그 미션의 준비 셀부터 바로 앞 STEP까지 순서대로 실행해 둡니다.</p><p>이번 입력 표는 <code>step2</code>, 완성한 결과 표는 <code>step3</code>입니다. 새로 만들 열은 <code>승인_미완료</code>이며 배정 시간은 36분입니다. 노트북의 <code>M07 STEP 03</code> 코드 셀에서 TODO 호출을 본인이 작성한 pandas 계산으로 바꾸세요. 결과는 DataFrame이어야 합니다.</p><h4>입력값의 뜻</h4><ul><li><code>status</code> — EXCLUDE_REQUESTED는 임무 조정 요청 후 아직 최종 승인이 끝나지 않은 상태입니다.</li></ul><h4>수행 순서</h4><ol><li>step2에서 status가 EXCLUDE_REQUESTED인지 확인합니다.</li><li>해당 상태에만 승인_미완료=True를 넣습니다.</li><li>새 열을 포함한 표를 step3에 저장합니다.</li><li>True인 건의 요청 내용·남은 승인 역할을 실행 기록에서 찾아 적습니다. 자료에 없는 승인 시각은 미확인으로 남깁니다.</li></ol><h4>결과를 읽는 방법</h4><p>True는 승인이 남아 있다는 표시입니다. 신병이나 프로그램이 임무 조정 결정을 대신 확정하라는 뜻이 아닙니다.</p><h4>직접 확인할 가상 예시</h4><p>다음은 규칙을 이해하기 위한 가상 사례입니다. 실제 실행값을 대신 제출하는 자료가 아닙니다.</p><table><thead><tr><th>가정한 입력</th><th>기대 결과</th><th>이유</th></tr></thead><tbody><tr><td><code>{"status": "EXCLUDE_REQUESTED"}</code></td><td><code>True</code></td><td>요청만 있고 최종 승인은 남아 있습니다.</td></tr><tr><td><code>{"status": "PENDING"}</code></td><td><code>False</code></td><td>다른 검토 대기 상태입니다.</td></tr><tr><td><code>{"status": "CLOSED"}</code></td><td><code>False</code></td><td>현재 승인 대기 상태가 아닙니다.</td></tr></tbody></table><h4>완료 전 확인과 기록</h4><p>원행ID는 계산 전후 같은 자료를 찾는 번호입니다. 입력의 행 수·행 순서·인덱스·원래 열의 값을 유지하고, 앞 단계에서 만든 열도 모두 남깁니다. 각 행의 계산 결과는 새 열에 넣습니다. 실제 불리언 True·False와 문자열 "True"·"False"는 다릅니다. 값이 없다는 뜻의 결측값을 False나 0으로 대신 채우지 않습니다.</p><p>계산 뒤 해당 출력 열과 입력 열을 원행ID로 함께 확인합니다. 실제 자료에서 해당 사례가 있으면 최소 한 행의 입력·출력·판단 이유를 기록하고, 해당 사례가 없으면 0건 집계와 아래 가상 예시 시험 결과를 남깁니다. 관찰 내용은 5단계 완료 후 HTML의 해당 STEP 기록란에 옮기고 기록 JSON 저장을 눌러 보관합니다. HTML 입력란의 글은 페이지를 닫기 전에 저장해야 합니다.</p><h4>주의할 점</h4><p>지휘관 역할만으로 군의관 검토를 건너뛰어 High 휴식 결정을 확정한 시험은 차단되어야 합니다. 요청과 승인, 현장 이행을 구분합니다.</p>


> **교수자·해설용 FAIL 점검 포인트**: 모든 미완료를 승인_미완료로 표시하거나 요청 기록만으로 최종 승인되었다고 설명하면 FAIL입니다.

In [ ]:
# M07 STEP 03 — 임무 조정 요청과 최종 승인을 구분하기 (교수자·해설용 예시 풀이)
# 해설: status == 'EXCLUDE_REQUESTED'(승인 대기)만 True로 표시합니다.
step3 = step2.copy()
step3["승인_미완료"] = (step3["status"] == "EXCLUDE_REQUESTED").astype(bool)
display(step3[["원행ID", "승인_미완료"]].head(10))

### M07 STEP 04 — 최초 접수 때 시스템이 자동으로 종결한 건이 있는지 확인하기

<h4>이 단계에서 해결할 일</h4><p>사람의 검토가 필요한 결정을 프로그램이 먼저 끝내면 책임 소재가 흐려집니다. 이번 값은 최초 접수 시 시스템이 자동 종결한 수이며 이후 사람의 승인·반려 건수와 구분해야 합니다.</p><h4>시작 전 준비</h4><p>각 미션의 STEP 1~5는 앞 단계 결과를 이어서 사용합니다. WEL01_m07_student.ipynb의 첫 실행(환경 준비) 셀과 이 미션의 준비 셀을 순서대로 실행하면 현재 참고 프로그램의 결과가 raw라는 입력 표로 만들어집니다. PARAMETERS는 기본값을 유지합니다. 이 단계만 다시 실행하려면 그 미션의 준비 셀부터 바로 앞 STEP까지 순서대로 실행해 둡니다.</p><p>이번 입력 표는 <code>step3</code>, 완성한 결과 표는 <code>step4</code>입니다. 새로 만들 열은 <code>자동조치_차단</code>이며 배정 시간은 36분입니다. 노트북의 <code>M07 STEP 04</code> 코드 셀에서 TODO 호출을 본인이 작성한 pandas 계산으로 바꾸세요. 결과는 DataFrame이어야 합니다.</p><h4>입력값의 뜻</h4><ul><li><code>자동조치수</code> — 최초 등록 시점의 시스템 자동 종결 건수입니다. 현재 CLOSED 상태의 건수가 아닙니다.</li></ul><h4>수행 순서</h4><ol><li>step3의 자동조치수와 참고 실행의 최초 등록 기록을 확인합니다.</li><li>자동조치수가 0이면 자동조치_차단=True, 0보다 크면 False로 만듭니다.</li><li>step4에 저장한 뒤 최초 등록 기록과 결과를 원행ID로 연결합니다.</li><li>별도의 위반 시도 6건 차단 기록과 긴급 연락 경로 모의 시험을 확인합니다. 이 두 시험의 성공을 자동조치수 대신 사용하지 않습니다.</li></ol><h4>결과를 읽는 방법</h4><p>True는 최초 접수에서 자동 종결이 없었다는 뜻입니다. 모든 권한 통제와 실제 연락이 확인되었다는 뜻은 아닙니다.</p><h4>직접 확인할 가상 예시</h4><p>다음은 규칙을 이해하기 위한 가상 사례입니다. 실제 실행값을 대신 제출하는 자료가 아닙니다.</p><table><thead><tr><th>가정한 입력</th><th>기대 결과</th><th>이유</th></tr></thead><tbody><tr><td><code>{"자동조치수": 0}</code></td><td><code>True</code></td><td>최초 자동 종결이 없습니다.</td></tr><tr><td><code>{"자동조치수": 1}</code></td><td><code>False</code></td><td>자동 종결이 한 건 있습니다.</td></tr></tbody></table><h4>완료 전 확인과 기록</h4><p>원행ID는 계산 전후 같은 자료를 찾는 번호입니다. 입력의 행 수·행 순서·인덱스·원래 열의 값을 유지하고, 앞 단계에서 만든 열도 모두 남깁니다. 각 행의 계산 결과는 새 열에 넣습니다. 실제 불리언 True·False와 문자열 "True"·"False"는 다릅니다. 값이 없다는 뜻의 결측값을 False나 0으로 대신 채우지 않습니다.</p><p>계산 뒤 해당 출력 열과 입력 열을 원행ID로 함께 확인합니다. 실제 자료에서 해당 사례가 있으면 최소 한 행의 입력·출력·판단 이유를 기록하고, 해당 사례가 없으면 0건 집계와 아래 가상 예시 시험 결과를 남깁니다. 관찰 내용은 5단계 완료 후 HTML의 해당 STEP 기록란에 옮기고 기록 JSON 저장을 눌러 보관합니다. HTML 입력란의 글은 페이지를 닫기 전에 저장해야 합니다.</p><h4>주의할 점</h4><p>음수나 누락된 건수는 올바른 입력이 아닙니다. 0으로 바꾸지 말고 확인받습니다. 긴급 연락 경로 시험은 코드 안의 모의 시험이며 실제 외부 연락이 아닙니다.</p>


> **교수자·해설용 FAIL 점검 포인트**: 현재 CLOSED 수를 자동조치수로 대신 쓰거나 이후 사람의 정상 종결을 자동 결정으로 세거나 모의 연락을 실제 연락으로 쓰면 FAIL입니다.

In [ ]:
# M07 STEP 04 — 최초 접수 때 시스템이 자동으로 종결한 건이 있는지 확인하기 (교수자·해설용 예시 풀이)
# 해설: 자동조치수가 결측/음수가 아닌 0일 때만 자동조치_차단=True입니다.
step4 = step3.copy()
if step4["자동조치수"].isna().any() or (step4["자동조치수"] < 0).any():
    raise ValueError("자동조치수에 결측 또는 음수가 있습니다.")
step4["자동조치_차단"] = (step4["자동조치수"] == 0).astype(bool)
display(step4[["원행ID", "자동조치_차단"]].head(10))

### M07 STEP 05 — 다음 담당자가 계속 맡아야 할 검토 건 정리하기

<h4>이 단계에서 해결할 일</h4><p>교대자가 미완료 결정을 놓치지 않도록 검토 대기와 승인 대기 표시를 한 목록으로 묶습니다. 누가 무엇을 계속 확인할지까지 적어야 다음 업무로 이어질 수 있습니다.</p><h4>시작 전 준비</h4><p>각 미션의 STEP 1~5는 앞 단계 결과를 이어서 사용합니다. WEL01_m07_student.ipynb의 첫 실행(환경 준비) 셀과 이 미션의 준비 셀을 순서대로 실행하면 현재 참고 프로그램의 결과가 raw라는 입력 표로 만들어집니다. PARAMETERS는 기본값을 유지합니다. 이 단계만 다시 실행하려면 그 미션의 준비 셀부터 바로 앞 STEP까지 순서대로 실행해 둡니다.</p><p>이번 입력 표는 <code>step4</code>, 완성한 결과 표는 <code>step5</code>입니다. 새로 만들 열은 <code>책임_인계필요</code>이며 배정 시간은 36분입니다. 노트북의 <code>M07 STEP 05</code> 코드 셀에서 TODO 호출을 본인이 작성한 pandas 계산으로 바꾸세요. 결과는 DataFrame이어야 합니다.</p><h4>입력값의 뜻</h4><ul><li><code>검토_대기</code> — STEP 1의 미완료 세 상태 표시입니다.</li><li><code>승인_미완료</code> — STEP 3의 임무 조정 승인 대기 표시입니다.</li></ul><h4>수행 순서</h4><ol><li>step4에서 두 표시를 읽습니다.</li><li>둘 중 하나라도 True이면 책임_인계필요=True로 하여 step5에 저장합니다.</li><li>True인 건마다 원행ID·상태·상급 검토 요청 유무·남은 승인·다음 담당 역할·확인 예정 시점을 기록합니다.</li><li>종결 건은 승인·반려 기록과 현장 이행 확인이 서로 다른 증거임을 적습니다. 확인되지 않은 이행을 완료로 쓰지 않습니다.</li></ol><h4>결과를 읽는 방법</h4><p>True는 책임을 이어받을 담당자가 필요한 건입니다. 실제 전달 완료는 전달 시각과 담당자 확인이 있어야 별도로 말할 수 있습니다.</p><h4>직접 확인할 가상 예시</h4><p>다음은 규칙을 이해하기 위한 가상 사례입니다. 실제 실행값을 대신 제출하는 자료가 아닙니다.</p><table><thead><tr><th>가정한 입력</th><th>기대 결과</th><th>이유</th></tr></thead><tbody><tr><td><code>{"검토_대기": true, "승인_미완료": false}</code></td><td><code>True</code></td><td>보류 등의 미완료 검토를 넘깁니다.</td></tr><tr><td><code>{"검토_대기": true, "승인_미완료": true}</code></td><td><code>True</code></td><td>승인 대기를 포함합니다.</td></tr><tr><td><code>{"검토_대기": false, "승인_미완료": false}</code></td><td><code>False</code></td><td>두 미완료 사유가 없습니다.</td></tr></tbody></table><h4>완료 전 확인과 기록</h4><p>원행ID는 계산 전후 같은 자료를 찾는 번호입니다. 입력의 행 수·행 순서·인덱스·원래 열의 값을 유지하고, 앞 단계에서 만든 열도 모두 남깁니다. 각 행의 계산 결과는 새 열에 넣습니다. 실제 불리언 True·False와 문자열 "True"·"False"는 다릅니다. 값이 없다는 뜻의 결측값을 False나 0으로 대신 채우지 않습니다.</p><p>계산 뒤 해당 출력 열과 입력 열을 원행ID로 함께 확인합니다. 실제 자료에서 해당 사례가 있으면 최소 한 행의 입력·출력·판단 이유를 기록하고, 해당 사례가 없으면 0건 집계와 아래 가상 예시 시험 결과를 남깁니다. 관찰 내용은 5단계 완료 후 HTML의 해당 STEP 기록란에 옮기고 기록 JSON 저장을 눌러 보관합니다. HTML 입력란의 글은 페이지를 닫기 전에 저장해야 합니다.</p><h4>주의할 점</h4><p>승인_미완료는 현재 정의상 검토_대기에 포함되지만 두 열을 보존하여 어떤 절차가 남았는지 보여 줍니다. 긴급 도움 요청은 비응급 승인을 기다리는 절차와 분리합니다.</p>


> **교수자·해설용 FAIL 점검 포인트**: AND를 사용해 보류 건을 빠뜨리거나 담당 역할·남은 확인 항목 없이 완료라고만 기록하면 FAIL입니다.

In [ ]:
# M07 STEP 05 — 다음 담당자가 계속 맡아야 할 검토 건 정리하기 (교수자·해설용 예시 풀이)
# 해설: 검토_대기 또는 승인_미완료 중 하나라도 True(OR)이면 책임_인계필요=True입니다.
step5 = step4.copy()
step5["책임_인계필요"] = (step5["검토_대기"] | step5["승인_미완료"]).astype(bool)
display(step5[["원행ID", "책임_인계필요"]].head(10))

### 마무리: 결과 화면 생성과 기록 저장

아래 셀을 실행하면 다섯 단계의 실제 계산 결과가 HTML로 저장됩니다. 왼쪽 STEP 메뉴로 결과를 확인하고 관찰 기록을 입력하세요. 기록 JSON을 내려받아야 입력한 글을 별도로 보관할 수 있습니다. 구조 검사 완료는 교관의 최종 합격 판정이 아닙니다.


In [ ]:
STAGE_SPECS = SOURCE_BUNDLE["missions"][MID]["stages"]
result_html = finish_mission(MID, raw, [step1, step2, step3, step4, step5], STAGE_SPECS)
